# Mode B3: Memetic + Two-Phase Strategy

**Enhancement**: Switches from exploration to exploitation mid-run.

| Phase | Generations | Repair Prob | Iterations | Rationale |
|-------|-------------|-------------|------------|----------|
| **1 (Exploration)** | 0-200 | 20% | 3 | Explore search space |
| **2 (Exploitation)** | 200+ | 50% | 10 | Intensive local refinement |

## 1. Imports

In [ ]:
from __future__ import annotations
import random, copy, time
import numpy as np
from pathlib import Path
from datetime import datetime
from dataclasses import dataclass, field

from deap import base, creator, tools
from schedule_engine.notebooks.core import (
    load_data, create_random_individual, course_aware_crossover, smart_mutation,
    create_evaluator, get_constraint_breakdown, setup_deap, get_best_individual, 
    EvolutionStats, print_constraint_details
)
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from schedule_engine.domain.gene import SessionGene
from schedule_engine.domain.types import SchedulingContext

from schedule_engine.ga.operators.repair import (
    repair_instructor_availability,
    repair_group_overlaps,
    repair_room_overlap_reassign,
    repair_room_conflicts,
    repair_instructor_conflicts,
    repair_instructor_qualifications,
    repair_room_type_mismatches,
)

print(" Imports successful")

## 2. Mode B3 Configuration

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 10
NGEN = 4000
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -1.0)

# MODE B3: Two-phase parameters
PHASE_SWITCH_GEN = 200  # Switch from exploration to exploitation

# Phase 1: Exploration (Gen 0-199)
PHASE1_REPAIR_PROB = 0.2
PHASE1_REPAIR_ITERATIONS = 3

# Phase 2: Exploitation (Gen 200+)
PHASE2_REPAIR_PROB = 0.5
PHASE2_REPAIR_ITERATIONS = 10

# Paths
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_b3_two_phase/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Mode B3: Two-phase strategy")
print(f"   Phase 1 (Gen 0-{PHASE_SWITCH_GEN-1}): prob={PHASE1_REPAIR_PROB}, iter={PHASE1_REPAIR_ITERATIONS}")
print(f"   Phase 2 (Gen {PHASE_SWITCH_GEN}+): prob={PHASE2_REPAIR_PROB}, iter={PHASE2_REPAIR_ITERATIONS}")

## 3. Load Data

In [ ]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

context = data.context
evaluate = create_evaluator(data)

print(f" {data.summary()}")

## 4. Repair Operator Wrapper

In [ ]:
@dataclass
class RepairStats:
    """Track repair operator statistics."""
    total_fixes: int = 0
    by_operator: dict[str, int] = field(default_factory=dict)


def apply_repair_operators(
    individual: list[SessionGene],
    context: SchedulingContext,
    max_iterations: int = 3,
) -> RepairStats:
    """Apply constraint-aware repair operators in priority order."""
    stats = RepairStats()
    
    repair_operators = [
        ("instructor_availability", repair_instructor_availability),
        ("group_overlaps", repair_group_overlaps),
        ("room_overlap_reassign", repair_room_overlap_reassign),
        ("room_conflicts", repair_room_conflicts),
        ("instructor_conflicts", repair_instructor_conflicts),
        ("instructor_qualifications", repair_instructor_qualifications),
        ("room_type_mismatches", repair_room_type_mismatches),
    ]
    
    for _ in range(max_iterations):
        iteration_fixes = 0
        for name, operator in repair_operators:
            try:
                fixes = operator(individual, context)
                if fixes > 0:
                    stats.by_operator[name] = stats.by_operator.get(name, 0) + fixes
                    stats.total_fixes += fixes
                    iteration_fixes += fixes
            except Exception:
                pass
        
        if iteration_fixes == 0:
            break
    
    return stats


print(" Repair operators ready")

## 5. Memetic NSGA-II with Two-Phase Strategy (Mode B3)

In [ ]:
LOG_INTERVAL = 10

def run_two_phase_nsga2():
    """Run NSGA-II with two-phase strategy."""
    print(f" Mode B3: Two-phase strategy, switch at gen {PHASE_SWITCH_GEN}")
    start = time.time()
    
    setup_deap(FITNESS_WEIGHTS)
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    total_repairs = 0
    phase1_repairs = 0
    phase2_repairs = 0
    
    for gen in range(NGEN):
        # === MODE B3: Phase-dependent parameters ===
        if gen < PHASE_SWITCH_GEN:
            repair_prob = PHASE1_REPAIR_PROB
            repair_iterations = PHASE1_REPAIR_ITERATIONS
            current_phase = 1
        else:
            repair_prob = PHASE2_REPAIR_PROB
            repair_iterations = PHASE2_REPAIR_ITERATIONS
            current_phase = 2
        
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # Repair with phase-dependent parameters
        for ind in offspring:
            if random.random() < repair_prob:
                repair_stats = apply_repair_operators(list(ind), context, repair_iterations)
                total_repairs += repair_stats.total_fixes
                if current_phase == 1:
                    phase1_repairs += repair_stats.total_fixes
                else:
                    phase2_repairs += repair_stats.total_fixes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        if gen % LOG_INTERVAL == 0 or gen == NGEN - 1 or gen == PHASE_SWITCH_GEN:
            phase_marker = "[SWITCH]" if gen == PHASE_SWITCH_GEN else f"[P{current_phase}]"
            best_ind = min(pop, key=lambda ind: (ind.fitness.values[0], ind.fitness.values[1]))
            breakdown = get_constraint_breakdown(list(best_ind), data)
            hard_names = {'student_group_exclusivity', 'instructor_exclusivity', 'instructor_qualifications', 
                         'room_suitability', 'room_exclusivity', 'instructor_time_availability', 
                         'room_time_availability', 'course_completeness'}
            hard_bd = {k: v for k, v in breakdown.items() if k in hard_names}
            soft_bd = {k: v for k, v in breakdown.items() if k not in hard_names}
            print_constraint_details(hard_bd, soft_bd, gen)
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s")
    print(f"   Phase 1 repairs: {phase1_repairs}")
    print(f"   Phase 2 repairs: {phase2_repairs}")
    print(f"   Total repairs: {total_repairs}")
    return pop, stats

final_pop, stats = run_two_phase_nsga2()

## 6. Results & Visualization

In [ ]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_b3_convergence.png", title_prefix="Mode B3: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_b3_breakdown.png", title="Mode B3: Constraint Violations")

## 7. Export Results

In [ ]:
from schedule_engine.notebooks.export import export_full_results

export_paths = export_full_results(
    population=final_pop,
    stats=stats,
    data=data,
    output_dir=OUTPUT_DIR,
    mode_name="mode_b3_two_phase",
)

print(f"\n All files saved to: {OUTPUT_DIR}")